Simulasi Roleplay: Tim Bisnis & Marketing vs. Data Analyst 

Halo Derick, selamat bergabung di tim! Kebetulan bulan ini kita sedang gencar untuk mengejar target kuartalan. Kami berencana meluncurkan dua kampanye (campaign) besar minggu depan:

Campaign A (Produk Proteksi Edukasi Syariah): Kami menyasar orang tua muda yang punya kesadaran finansial tinggi agar mereka menyiapkan tabungan pendidikan anak berbasis syariah.

Campaign B (Win-Back Program untuk Nasabah Lapsed): Kami ingin mendekati kembali nasabah lama yang polisnya baru saja hangus (lapsed) agar mereka mau mengaktifkan kembali polisnya dengan promo bebas biaya administrasi.

Tim kami butuh bantuanmu untuk menarik data mentah dari database. Tolong siapkan beberapa file data berikut dalam format Excel (.xlsx) ya, jangan CSV agar anak-anak marketing bisa langsung filter manual di Excel mereka

(Tidak ada AI yang digunakan, AI hanya digunakan untuk pembuatan simulasi dan pencarian dataset. Proses pencarian solusi pure berdasarkan W3School dan sumber internet lainnya!)

In [35]:
#Source of Dataset = https://www.kaggle.com/datasets/janiobachmann/bank-marketing-dataset
import pandas as pd

data = pd.read_csv('datasets/bank_marketing_dataset.csv')

Request 1: Leads Data untuk Campaign A (Edukasi Syariah)
Tolong carikan data nasabah kita yang memenuhi kriteria berikut:

1. Berusia antara 27 sampai 42 tahun (asumsi kami mereka punya anak usia sekolah).
2. Status pernikahan: Menikah (Married).
3. Pekerjaan: Fokus ke yang bukan Blue-collar (pilih manajemen, teknisi, admin, atau entrepreneur) karena produk ini preminya lumayan tinggi.
4. Tolong keluarkan/exclude nasabah yang saat ini sudah memiliki saldo tabungan/dana di bawah $300 (+- Rp.5.000.000)  (kita cari yang ekonominya stabil).
5. Output kolom yang kami butuh: Umur, Pekerjaan, Kontak/No HP, dan Kota.

In [29]:
#Solution for Request 1 
filtered_data = data[(data['age']>= 27) & (data['age']<= 42) & (data['marital']=='married') & (data['job']!='blue-collar') & (data['balance']>300)]
data_output = filtered_data[['age','job','contact','balance']]

data_output.to_excel('req1_solution.xlsx',sheet_name='campaignA_req1',index=False)

Request 2: Evaluasi Target Campaign B (Win-Back Program)
Kami butuh data nasabah yang polisnya tercatat 'Lapsed' atau gagal bayar dalam kurun waktu 3 bulan terakhir. Namun, agar efisien:

Filter hanya nasabah yang di campaign sebelumnya memberikan respons positif atau minimal status poutcome-nya (outcome sebelumnya) adalah success atau unknown (jangan ambil yang terang-terangan menolak atau failure di masa lalu).

Output kolom yang kami butuh: ID Nasabah, Jenis Produk Terakhir, Kontak/No HP, Tanggal Lapsed, dan Sisa Saldo Investasi Terakhir.

In [30]:
#Karena gaada kolom lapsed, kita alihkan jadi data dengan previous outcome 
# berhasil(berhasil convert dicampaign lalu dan memiliki balance >300$)

filter = data[(data['balance']>=300) & (~data['poutcome'].isin(['failure','unknown']))]
data_final=filter[['contact','previous','balance']]
data_final.to_excel('req2_solution.xlsx',sheet_name='campaignA_req1',index=False)

Request 3: Summary Singkat untuk Bahan Meeting (Tanpa Grafik)
Sebelum kami eksekusi, tolong buatkan summary text sederhana di Jupyter Notebook-mu (bisa di-print saja kodenya) yang menunjukkan Top 3 tipe pekerjaan dan Top 3 kota dari nasabah kita yang paling sering bilang "Yes" (membeli produk) pada campaign bulan lalu. Ini buat bahan argumen kami ke Direksi.

In [33]:
filter_data =data[(data['poutcome']==('success'))]
filter_data=filter_data
print(filter_data['job'].value_counts().head(3))

job
management    292
technician    157
retired       135
Name: count, dtype: int64


SIMULASI MODELING PREDIKSI RISIKO ASURANSI USING XGBoost and Random Forest

Dataset used : https://www.kaggle.com/c/prudential-life-insurance-assessment/data 

In [ ]:
#XGBoost 
from sklearn.model_selection import train_test_split 
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import cohen_kappa_score

train = pd.read_csv('datasets/prudential_assessment/train.csv')
test = pd.read_csv('datasets/prudential_assessment/test.csv')

X= train.drop(columns=['Id','Response'])
Y= train['Response']-1

X=X.fillna(X.median(numeric_only=True))
for col in X.select_dtypes('object').columns:
    X[col]=X[col].astype('category').cat.codes

X_train,X_val,Y_train,Y_val=train_test_split(X,Y,test_size=0.2, random_state=42,stratify=Y)

model = XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
model.fit(X_train,Y_train)

train_prediction = model.predict(X_train)
val_prediction = model.predict(X_val)

cks = cohen_kappa_score(Y_val,val_prediction,weights='quadratic')
print(classification_report(Y_val,val_prediction))
print(f'cks quadratic score : {cks}')

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [00:41:05] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "n_job" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


              precision    recall  f1-score   support

           0       0.49      0.22      0.31      1241
           1       0.48      0.26      0.34      1310
           2       0.56      0.41      0.48       203
           3       0.60      0.70      0.65       286
           4       0.60      0.54      0.57      1086
           5       0.53      0.53      0.53      2247
           6       0.48      0.44      0.46      1606
           7       0.66      0.92      0.77      3898

    accuracy                           0.59     11877
   macro avg       0.55      0.50      0.51     11877
weighted avg       0.57      0.59      0.56     11877

cks quadratic score : 0.5437245510748014


In [ ]:
X_test = test.drop(columns=['Id'])
X_test = X_test.fillna(X_test.median(numeric_only=True))

for col in X_test.select_dtypes('object').columns:
    X_test[col] = X_test[col].astype('category').cat.codes

test_prediction = model.predict(X_test)
final_predictions = test_prediction + 1

submission = pd.DataFrame({
    'Id': test['Id'],
    'Response': final_predictions
})

testData = pd.read_csv('datasets/prudential_assessment/test.csv')
testDataPrudential = pd.merge(testData,submission,on='Id',how='inner')
testDataPrudential.to_csv('solutions/DataPrudentialForVisualization.csv',index=False)

In [ ]:
# Random Forest

model2=RandomForestClassifier(n_estimators=100,random_state=42) 
model2.fit(X_train,Y_train)

train_prediction2=model2.predict(X_train)
val_prediction2=model2.predict(X_val)

cks=cohen_kappa_score(Y_val,val_prediction2,weights='quadratic')

print(classification_report(Y_val,val_prediction2))
print(f'cohen_kappa_score:{cks}')